# Course 9: Mathematics and AI
## Session 10: AI for theorem proving

**Goal:** understand proof search and the pass@k metric with a small runnable model, then plan an evaluation of an open prover on miniF2F.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. Proof search in miniature
A prover is a search: states are goals, actions are tactics, and a checker says whether a step is legal. Here is a toy with the same shape. The 'goal' is to turn a start number into a target using a few moves, and a policy scores which move to try first. A good policy finds short 'proofs' with few checker calls.

In [ ]:
import heapq, random, numpy as np
MOVES = {"add 3": lambda x: x + 3, "double": lambda x: x * 2, "subtract 1": lambda x: x - 1}

def search(start, target, policy, budget=200, limit=500):
    """Best-first search. Returns (proof or None, checker calls used)."""
    frontier = [(0.0, 0, start, [])]; seen = {start}; calls = 0; tie = 0
    while frontier and calls < budget:
        cost, _, state, path = heapq.heappop(frontier)
        if state == target: return path, calls
        for name, fn in MOVES.items():
            calls += 1; nxt = fn(state)                       # one call to the 'proof checker'
            if nxt in seen or not (0 <= nxt <= limit): continue
            seen.add(nxt); tie += 1
            heapq.heappush(frontier, (cost + 1 - policy(nxt, target), tie, nxt, path + [name]))
    return None, calls

blind = lambda state, target: 0.0                               # breadth-first: no guidance
guided = lambda state, target: 1.0 / (1.0 + abs(target - state) / 10)       # prefers states near the target
problems = [(random.Random(i).randint(1, 9), random.Random(100 + i).randint(60, 300)) for i in range(50)]
for name, pol in [("no guidance", blind), ("guided policy", guided)]:
    res = [search(s, t, pol) for s, t in problems]
    solved = [r for r in res if r[0] is not None]
    print(f"{name:14s} solved {len(solved)}/50 within 200 checker calls | mean calls when solved {np.mean([c for _, c in solved]):.0f}")
s, t = next((s, t) for s, t in problems if search(s, t, guided)[0]); p, c = search(s, t, guided)
print(f"a proof of {s} -> {t} found with {c} checker calls:", p)
assert p is not None

Real provers replace the three moves with thousands of possible tactics generated by a language model, and the checker with Lean. The budget of checker calls is what 'compute' means in prover papers.

### 2. pass@k
A prover is run k times per problem (different samples), and a problem counts as solved if any attempt is accepted by Lean. Since Lean rejects wrong proofs, more attempts can only help, and the cost grows with k. Always report k.

In [ ]:
rng = np.random.default_rng(0)
p_solve = rng.beta(0.35, 1.2, size=244)                       # an imagined per-attempt success chance for each of 244 problems
def pass_at_k(p, k): return float(np.mean(1 - (1 - p) ** k))
for k in [1, 8, 32, 128, 1024]: print(f"pass@{k:<5d} {pass_at_k(p_solve, k):.3f}")
print("share of problems with under 1% chance per attempt:", round(float((p_solve < 0.01).mean()), 2))

The curve flattens: a core of problems is nearly never solved, and thousands more attempts barely move the number. Comparing a system at pass@32 with another at pass@8192 says more about budgets than about ideas.

### 3. Estimating pass@k from n samples
If you drew n attempts per problem and c were correct, the unbiased estimate of pass@k (for k <= n) is `1 - C(n - c, k) / C(n, k)`. Do not estimate it by looking only at the first k attempts: that wastes data and is noisier.

In [ ]:
from math import comb
def pass_at_k_estimate(n, c, k): return 1.0 - comb(n - c, k) / comb(n, k)
n = 64; attempts = rng.random((244, n)) < p_solve[:, None]; c = attempts.sum(1)
for k in [1, 8, 32]:
    unbiased = np.mean([pass_at_k_estimate(n, ci, k) for ci in c]); naive = attempts[:, :k].any(1).mean()
    print(f"pass@{k:<3d} true {pass_at_k(p_solve, k):.3f} | unbiased estimate {unbiased:.3f} | first-k estimate {naive:.3f}")
est = np.mean([pass_at_k_estimate(n, ci, 8) for ci in c]); assert abs(est - pass_at_k(p_solve, 8)) < 0.05

### 4. An error bar
50 problems is a small test. Bootstrap over problems.

In [ ]:
solved = (rng.random(50) < 0.4).astype(float)
boots = rng.choice(solved, (5000, 50)).mean(1)
print(f"solved {int(solved.sum())}/50 = {solved.mean():.2f}   95% interval {np.percentile(boots, 2.5):.2f} to {np.percentile(boots, 97.5):.2f}")

With 50 problems the interval is about ±14 points. Two provers that differ by 3 problems are not distinguishable on this subset.

### 5. Run an open prover on 50 miniF2F problems (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU, a Lean 4 toolchain, and LeanDojo (or another Lean-Python bridge such as the Lean REPL or Pantograph).
# Interfaces in this area change quickly: read the current README of the tool you choose before copying anything below.
#
# Plan
# 1. Problems: take 50 problems from the miniF2F test split (Lean 4 version). Fix the list and the Mathlib version, and write both down.
# 2. Prover: an open whole-proof model from the Hugging Face hub (for example a DeepSeek-Prover or Goedel-Prover checkpoint that fits your GPU).
# 3. For each problem, sample k complete proofs (start with k = 8) at temperature 1.0.
# 4. Check each proof with Lean. A proof counts only if it compiles with no errors and contains no `sorry`.
# 5. Report pass@1 and pass@8 with a bootstrap interval, the wall-clock time, and the number of Lean checks.
#
# def check(statement: str, proof: str) -> bool:
#     source = HEADER + statement + " := by\n" + proof
#     result = lean_repl.run(source, timeout=120)                # depends on your bridge
#     return result.ok and "sorry" not in proof
#
# Common problems: Mathlib version mismatch between the model's training data and your install (lemma names differ);
# timeouts counted as failures (say so); statements that are accidentally trivial or false in the benchmark (report any you find).

### TODO
1. In section 1, write a better policy and beat the guided one on mean checker calls.
2. Run the miniF2F evaluation and fill in: pass@1, pass@8, interval, time.
3. Pick 3 failed problems. For each: was the proof idea wrong, or did a correct idea fail on syntax or a renamed lemma? This split matters more than the headline number.